In [1]:
# Setup the Jupyter version of Dash
from dash import Dash

# Configure the necessary Python module imports for dashboard components
import dash
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
from pymongo.errors import PyMongoError

# Configure the plotting routines
import pandas as pd

# Imports data from module
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU1234"
host = 'localhost'
port = 27017
database = 'AAC'
collection = 'animals'

# Connect to database via CRUD Module
shelter = AnimalShelter(username, password, host, port, database, collection)

###########################
# Enhancement Three
# Schema Validation Tests
###########################

schema_result = shelter.configure_schema_validation()
print("Schema validation result:", schema_result)

collection_info = shelter.database.command(
    "listCollections",
    filter={"name": "animals"}
)

options = collection_info["cursor"]["firstBatch"][0].get("options", {})

print("Validator:")
print(options.get("validator"))

print("\nValidation Level:")
print(options.get("validationLevel"))

print("\nValidation Action:")
print(options.get("validationAction"))

# test 1 invalid age data type
invalid_animal = {
    "rec_num": 20001,
    "name": "Database Test",
    "animal_type": "Dog",
    "breed": "Labrador",
    "age_upon_outcome_in_weeks": "five",
    "sex_upon_outcome": "Neutered Male"
}

try:
    shelter.collection.insert_one(invalid_animal)
    print("TEST 1 FAILED: Invalid record was inserted.")
except PyMongoError:
    print("TEST 1 PASSED: Invalid age was rejected by MongoDB.")

# test 2 Missing required field
missing_field_animal = {
    "rec_num": 20002,
    "name": "Database Test",
    "animal_type": "Dog",
    # breed intentionally missing for test
    "age_upon_outcome_in_weeks": 52,
    "sex_upon_outcome": "Neutered Male"
}

try:
    shelter.collection.insert_one(missing_field_animal)
    print("TEST 2 FAILED: Record with missing field was inserted.")
except PyMongoError:
    print("TEST 2 PASSED: Missing required field was rejected by MongoDB.")


# test 3 invalid sex value
invalid_sex_animal = {
    "rec_num": 20003,
    "name": "Database Test",
    "animal_type": "Dog",
    "breed": "Labrador",
    "age_upon_outcome_in_weeks": 52,
    "sex_upon_outcome": "Male"
}

try:
    shelter.collection.insert_one(invalid_sex_animal)
    print("TEST 3 FAILED: Invalid sex value was inserted.")
except PyMongoError:
    print("TEST 3 PASSED: Invalid sex value was rejected by MongoDB.")


# test 4 valid record
valid_animal = {
    "rec_num": 20004,
    "name": "Database Test",
    "animal_type": "Dog",
    "breed": "Labrador",
    "age_upon_outcome_in_weeks": 52,
    "sex_upon_outcome": "Neutered Male"
}

try:
    result = shelter.collection.insert_one(valid_animal)

    if result.inserted_id:
        print("TEST 4 PASSED: Valid animal was accepted by MongoDB.")

except PyMongoError:
    print("TEST 4 FAILED: Valid animal was rejected by MongoDB.")

# test 5 CRUD create method handles schema validation failure
invalid_create = {
    "name": "CRUD Test",
    "animal_type": "Dog",
    "breed": "Labrador",
    "age_upon_outcome_in_weeks": "invalid age",
    "sex_upon_outcome": "Neutered Male"
}

create_result = shelter.create(invalid_create)

if create_result is False:
    print("TEST 5 PASSED: CRUD create method handled invalid data correctly.")
else:
    print("TEST 5 FAILED: Invalid data was accepted.")

    
test_record = shelter.collection.find_one(
    {"rec_num": {"$exists": True, "$type": "int"}}
)

print(test_record)

# test 6 CRUD update method handles schema validation failure

update_result = shelter.update(
    {"rec_num": 10003},
    {"age_upon_outcome_in_weeks": "invalid age"}
)

if update_result == 0:
    print("TEST 6 PASSED: CRUD update method rejected invalid data.")
else:
    print("TEST 6 FAILED: Invalid update was accepted.")

# Verify original value was preserved
verify_record = shelter.collection.find_one({"rec_num": 10003})

print(
    "Age after failed update:",
    verify_record["age_upon_outcome_in_weeks"]
)

###########################
# End of Testing
###########################

# Class read method must support return of list object
# and accept projection JSON input.
# Sending the read method an empty document requests
# all documents be returned.
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB returns the '_id' column as an ObjectId.
# Dash DataTable cannot properly display ObjectId values,
# so the column is removed from the DataFrame.
df.drop(columns=['_id'], inplace=True)


#########################
# Dashboard Layout / View
#########################

app = Dash(__name__)


app.layout = html.Div([

    html.Div(id='hidden-div', style={'display': 'none'}),
    dcc.Store(id='refresh-data',data=0),
    html.H1('CS-340 Dashboard', style={'textAlign': 'center'}),
    html.Div(id="query-out", style={'whiteSpace': 'pre-line'}),
    html.H1("Kaitlin's Client Server Authentication Dashboard:", style={'textAlign': 'center'}),

    html.Hr(),

    ############################
    # Animal Management Interface
    ############################

    html.Div([

        html.H2("Animal Management"),

        # animal name
        html.Div([

            html.Div([
            html.Label("Animal Name:"),
            html.Br(),

            dcc.Input(id='animal-name', type='text', placeholder='Enter animal name', style={'width': '200px'})              
        ]),


        # animal type
        html.Div([
            html.Label("Animal Type:"),

            dcc.Dropdown(
                id='animal-type',
                options=[
                    {'label': 'Dog','value': 'Dog'},
                    {'label': 'Cat','value': 'Cat'},
                    {'label': 'Other','value': 'Other'}
                ],
                placeholder='Select animal type', value=None, clearable=True,
                style={'width': '200px'}
            )

        ]),


        # animal breed
        html.Div([
            html.Label("Animal Breed:"),
            html.Br(),

            dcc.Input(id='animal-breed', type='text', placeholder='Enter animal breed', style={'width': '200px'})

        ]),


        # animal age
        html.Div([
            html.Label("Animal Age (in weeks):"),
            html.Br(),

            dcc.Input(id='animal-age', type='number', min=0, placeholder='Enter animal age', style={'width': '200px'})

        ]),


        # animal sex
        html.Div([
            html.Label("Animal Sex Upon Outcome:"),

            dcc.Dropdown(
                id='animal-sex',
                options=[
                    {'label': 'Intact Male', 'value': 'Intact Male'},
                    {'label': 'Intact Female','value': 'Intact Female'},
                    {'label': 'Neutered Male','value': 'Neutered Male'},
                    {'label': 'Spayed Female','value': 'Spayed Female'},
                    {'label': 'Unknown','value': 'Unknown'}
                ],
                placeholder='Select animal sex', value=None, clearable=True,
                style={'width': '200px'}
            )

        ])

    ], style ={
        'display': 'flex',
        'gap': '20px',
        'alignItems': 'flex-end',
        'flexWrap': 'wrap',
        'marginBottom': '15px'
    }),

        ########################
        # Management Buttons
        ########################

        html.Button(
            "Add Animal",
            id="add-animal-btn",
            n_clicks=0
        ),

        html.Button(
            "Update Animal",
            id="update-animal-btn",
            n_clicks=0,
            style={'marginLeft': '10px'}
        ),

        html.Button(
            "Delete Animal",
            id="delete-animal-btn",
            n_clicks=0,
            style={'marginLeft': '10px'}
        ),

        html.Button(
            "Clear Form",
            id="clear-animal-btn",
            n_clicks=0,
            style={'marginLeft': '10px'}
        ),


        # Status Message
        html.Div(
            id='animal-management-status',
            style={'marginTop': '10px'}
        )

    ], style={
        'padding': '15px',
        'border': '1px solid #ccc',
        'marginBottom': '20px'
    }),

    ###############################
    # Advance Animal Search Display
    ###############################

html.Div([
        html.H2("Advanced Animal Search"),

        html.Div([

        # Type filter
        html.Div([
            html.Label("Animal Type:"),

            dcc.Dropdown(
                id='search-animal-type',
                options=[ {'label': 'Dog', 'value': 'Dog'},
                         {'label': 'Cat', 'value': 'Cat'},
                         {'label': 'Other', 'value': 'Other'}
                         ],
                         placeholder='Select Animal Type',
                         value=None,
                         clearable=True,
                         style={'width': '200px'}
            )
        ]),

        # Breed filter
        html.Div([
            html.Label("Breed"),
            html.Br(),

            dcc.Input(
                id='search-breed',
                type='text',
                placeholder='Enter Breed',
                style={'width': '200px'}
            )
        ]),

        # Sex filter
        html.Div([
            html.Label("Animal Sex:"),

            dcc.Dropdown(
                id='search-sex',
                options=[
                    {'label': 'Intact Male', 'value': 'Intact Male'},
                    {'label': 'Intact Female', 'value': 'Intact Female'},
                    {'label': 'Neutered Male', 'value': 'Neutered Male'},
                    {'label': 'Spayed Female', 'value': 'Spayed Female'},
                    {'label': 'Unknown', 'value': 'Unknown'}
                ],
                placeholder='Select Animal Sex',
                value=None,
                clearable=True,
                style={'width': '200px'}
            )
        ]),

        #Min value age
        html.Div([
            html.Label("Minimal Age (weeks):"),
            html.Br(),

            dcc.Input(
                id='search-min-age',
                type='number',
                min=0,
                placeholder='Minimum age',
                style={'width': '175px'}
            )
        ]),

        # Max value age
        html.Div([
            html.Label("Maximum Age (weeks):"),
            html.Br(),

            dcc.Input(
                id='search-max-age',
                type='number',
                min=0,
                placeholder='Maximum age',
                style={'width': '175px'}
            )
        ]),

        # Sort field
        html.Div([
            html.Label("Sort By:"),

            dcc.Dropdown(
                id='search-sort-by',
                options=[
                    {'label': 'Animal Name', 'value': 'name'},
                    {'label': 'Animal Type', 'value': 'animal_type'},
                    {'label': 'Breed', 'value': 'breed'},
                    {'label': 'Age', 'value': 'age_upon_outcome_in_weeks'}
                ],
                placeholder='Select sorting field',
                value=None,
                clearable=True,
                style={'width': '200px'}
            )
        ]),

        # Sort direction
        html.Div([
            html.Label("Sort Order:"),

            dcc.RadioItems(
                id='search-sort-order',
                options=[
                    {'label': 'Ascending', 'value': 'asc'},
                    {'label': 'Descending', 'value': 'desc'}
                ],
                value='asc',
                inline=True
            )
        ])

], style={
    'display': 'flex',
    'gap': '20px',
    'alignItems': 'flex-end',
    'flexWrap': 'wrap',
    'marginBottom': '15px'
}),

 # Search button
html.Button(
            "Search",
            id='search-btn',
            n_clicks=0
        ),

        # Reset button
html.Button(
            "Reset Search",
            id='reset-search-btn',
            n_clicks=0,
            style={'marginLeft': '10px'}
        ),

        # Search status
html.Div(
            id='search-status',
            style={'marginTop': '10px'}
        )
    ], style={
        'padding': '15px',
        'border': '1px solid #ccc',
        'marginBottom': '20px'
    }),

html.Hr(),

    ################################
    # Rescue Type Filtering Options
    ################################

    html.Div([
        html.Label("Filter By Rescue Type:"),

        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain or Wilderness'},
                {'label': 'Disaster or Individual Rescue', 'value': 'Disaster or Individual'},
                {'label': 'Reset', 'value': 'All'},
            ],
            value='All',
            inline=True
        )

    ], style={'padding': '10px'}),

    html.Hr(),

    ########################
    # Interactive DataTable
    ########################

    dash_table.DataTable(
       id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        editable=False,                               # Prevents user from editing cells except through the update form
        filter_action="native",                       # Column filtering
        sort_action="native",                         # Sort by column
        sort_mode="multi",                            # Sort multiple columns
        row_selectable="single",                      # Allows user to select a row
        selected_rows=[0],                            # Tracks selected row(s)
        row_deletable=False,                          # Prevents user from deleting row(s)
        column_selectable="single",                   # Allows user to select a column
        selected_columns=[],                          # Tracks selected column(s)
        page_action="native",                         # Built-in pagination, 10 rows/pg
        page_current=0,
        page_size=10,
    ),

    html.Br(),
    html.Hr(),


    ########################
    # Graph and Map
    ########################

    # Sets up the dashboard so that the chart
    # and geolocation map are side-by-side.
    html.Div(className='row', style={'display': 'flex'}, children=[
            html.Div(id='graph-id', className='col s12 m6'),
            html.Div(id='map-id', className='col s12 m6')
        ])
])


#############################################
# Interaction Between Components / Controller
#############################################


########################
# Query Filtering
########################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value'),
     Input('refresh-data','data'),
     Input('search-btn', 'n_clicks'),
     Input('reset-search-btn', 'n_clicks')],
     [State('search-animal-type', 'value'),
      State('search-breed', 'value'),
      State('search-sex', 'value'),
      State('search-min-age', 'value'),
      State('search-max-age', 'value'),
      State('search-sort-by', 'value'),
      State('search-sort-order', 'value')]
)

def update_dashboard(filter_type, refresh_data, search_clicks, reset_clicks, animal_type, breed, sex, min_age, max_age, sort_by, sort_order):
    query = {}

    # Determines what triggered the callback
    triggered_id=dash.ctx.triggered_id

    # Reset search
    if triggered_id == 'reset-search-btn':
        query = {}

    # Advance search query when search btn clicked
    elif triggered_id == 'search-btn':

        # Filter animal type
        if animal_type:
            query["animal_type"] = animal_type

        # Filter breed (case-insenstive)
        if breed:
            query["breed"] = {
                "$regex": breed.strip(),
                "$options": "i" 
            }

        # Filter sex
        if sex:
            query["sex_upon_outcome"] = sex

        # Filter age range
        age_query = {}

        if min_age is not None:
            age_query["$gte"] = min_age

        if max_age is not None:
            age_query["$lte"] = max_age

        if age_query:
            query["age_upon_outcome_in_weeks"] = age_query    

    elif filter_type == 'Water':
        query = {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26,"$lte": 156},
            "breed": {"$in": ["Labrador Retriever Mix","Chesa Bay Retr Mix","Newfoundland Mix"]}
        }

    elif filter_type == 'Mountain or Wilderness':
        query = {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26,"$lte": 156},
            "breed": {"$in": ["German Shepherd","Alaskan Malamute","Old English Sheepdog","Siberian Husky","Rottweiler"]}
        }

    elif filter_type == 'Disaster or Individual':
        query = {
            "animal_type": "Dog",
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20,"$lte": 300 },
            "breed": {"$in": ["Doberman Pinscher","German Shepherd","Golden Retriever","Bloodhound","Rottweiler"]}
        }

    else:
        query = {}

    # Retrieves matching records
    results = shelter.read(query)

    # Convert results into dataframe
    filtered_df = pd.DataFrame.from_records(results)

    # Removed MongoDB objectid before displaying data
    filtered_df.drop(columns=['_id'], inplace=True, errors='ignore')

    # Sort advanced search results based on selection
    if triggered_id =='search-btn' and sort_by and not filtered_df.empty:

        # Sorting direction
        ascending = sort_order == 'asc'

        # Sort results by selected field
        filtered_df = filtered_df.sort_values(
            by=sort_by,
            ascending=ascending,
            na_position='last' 
        )

    # Return filtered & sorted records to table
    return filtered_df.to_dict('records')

########################
# Reset Search
########################

@app.callback(
        [Output('search-animal-type', 'value'),
         Output('search-breed', 'value'),
         Output('search-sex', 'value'),
         Output('search-min-age', 'value'),
         Output('search-max-age', 'value'),
         Output('search-sort-by', 'value'),
         Output('search-sort-order', 'value')],
         Input('reset-search-btn', 'n_clicks'),
         prevent_initial_call=True
)

def reset_advanced_search(n_clicks):
    # Clears search criteria & restores default sort order
    return None, None, None, None, None, None, 'asc'

########################
# Pie Chart
########################

# Display the breeds of animals based on quantity
# represented in the DataTable.

@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)

def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return []

    df_view = pd.DataFrame(viewData)
    fig = px.pie(df_view,names='breed',title='Preferred Animals')

    return [dcc.Graph(figure=fig)]


########################
# Selected Column Styling
########################

# Highlights a column when the user selects it.

@app.callback(
    Output('datatable-id','style_data_conditional'),
    [Input('datatable-id','selected_columns')]
)

def update_styles(selected_columns):

    return [ {
            'if': {'column_id': i},
            'background_color': '#D2F3FF'
        } for i in selected_columns]

#########################################
# Selected animal management & clear form
##########################################

# Populates animal management form with selected row data
@app.callback([
    Output('animal-name', 'value'),
    Output('animal-type', 'value'),
    Output('animal-breed', 'value'),
    Output('animal-age', 'value'),
    Output('animal-sex', 'value')
],
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows'),
     Input('clear-animal-btn', 'n_clicks')]
)

def populate_animal_management_form(viewData, selected_rows, clear_clicks):

    # Determine what causes the callback
    triggered_id = dash.ctx.triggered_id

    # Clear all form fields when clear form button is clicked
    if triggered_id == 'clear-animal-btn':
        return [None, None, None, None, None]

    # If no row is selected, return None for all form fields
    if not viewData:
        return [None, None, None, None, None]

    # Use selected row or default to first row if no selection
    row_index = selected_rows[0] if selected_rows else 0

    # Make sure the selected row exists
    if row_index >= len(viewData):
        return [None, None, None, None, None]

    animal = viewData[row_index]

    return (
        animal.get('name'),
        animal.get('animal_type'),
        animal.get('breed'),
        animal.get('age_upon_outcome_in_weeks'),
        animal.get('sex_upon_outcome')
    )

########################
# Add Animal
########################

# Adds new animal record to DB

@app.callback(
    [Output('animal-management-status', 'children'),
    Output('refresh-data','data')],
    Input('add-animal-btn', 'n_clicks'),
    [State('animal-name', 'value'),
     State('animal-type', 'value'),
     State('animal-breed', 'value'),
     State('animal-age', 'value'),
     State('animal-sex', 'value'),
     State('refresh-data', 'data')],
     prevent_initial_call=True
)

def add_animal(n_clicks, name, animal_type, breed, age, sex, refresh_data):

    # Validate required fields before adding animal
    if n_clicks > 0:
        if not name or not animal_type or not breed or age is None or not sex:
            return "Please fill in all fields before adding an animal.", refresh_data

        # Create a new animal record
        new_animal = {
            "name": name.strip(),
            "animal_type": animal_type,
            "breed": breed,
            "age_upon_outcome_in_weeks": age,
            "sex_upon_outcome": sex
        }

        # Add the new animal to the database successfully
        success = shelter.create(new_animal)

        if success:
            return f"Animal '{name}' added successfully!", refresh_data + 1

        return "Error: Record did not meet validation requirements. Please enter valid information.", refresh_data

########################
# Update Animal
########################

# Updates selected animal record in DB

@app.callback(
    [Output('animal-management-status', 'children', allow_duplicate=True),
    Output('refresh-data','data', allow_duplicate=True)],
    Input('update-animal-btn', 'n_clicks'),

    [State('datatable-id', 'derived_virtual_data'),
     State('datatable-id', 'derived_virtual_selected_rows'),
     State('animal-name', 'value'),
     State('animal-type', 'value'),
     State('animal-breed', 'value'),
     State('animal-age', 'value'),
     State('animal-sex', 'value'),
     State('refresh-data', 'data')],

    prevent_initial_call=True)

def update_animal(n_clicks, viewData, selected_rows, name, animal_type, breed, age, sex, refresh_data):

    # Ensure a record is selected
    if not viewData or not selected_rows:
        return "Please select an animal record to update.", refresh_data

    # Validate required fields before updating animal
    if not name or not animal_type or not breed or age is None or not sex:
        return "Please fill in all fields before updating a record.", refresh_data

    # Get the selected animal record
    row_index = selected_rows[0]

    if row_index >= len(viewData):
        return "Error: Selected record could not be found.", refresh_data

    selected_animal = viewData[row_index]

    # Get the record number of the selected animal
    rec_num = selected_animal.get('rec_num')

    if rec_num is None:
        return "Error: No record number could be found.", refresh_data

    # Updated values
    updated_values = {
        "name": name.strip(),
        "animal_type": animal_type,
        "breed": breed.strip(),
        "age_upon_outcome_in_weeks": age,
        "sex_upon_outcome": sex
    }

    # Updated MongoDB
    modified_count = shelter.update({"rec_num": rec_num}, updated_values)

    if modified_count > 0:
        return f"Animal record '{name}' updated successfully!", refresh_data + 1

    return "No records were updated. Please check that values meet validation requirements.", refresh_data

########################
# Delete Animal
########################

# Deletes selected animal record in DB

@app.callback(
    [Output('animal-management-status', 'children', allow_duplicate=True),
    Output('refresh-data','data', allow_duplicate=True)],
    Input('delete-animal-btn', 'n_clicks'),

    [State('datatable-id', 'derived_virtual_data'),
     State('datatable-id', 'derived_virtual_selected_rows'),
     State('refresh-data', 'data')],

    prevent_initial_call=True)

def delete_animal(n_clicks, viewData, selected_rows, refresh_data):

    # Ensure a record is selected
    if not viewData or not selected_rows:
        return "Please select an animal record to delete.", refresh_data

    # Get the selected animal record
    row_index = selected_rows[0]

    # Make sure the selected row exists
    if row_index >= len(viewData):
        return "Error: Selected record could not be found.", refresh_data

    # Get the selected animal record
    selected_animal = viewData[row_index]

    # Get the record number of the selected animal
    rec_num = selected_animal.get('rec_num')

    if rec_num is None:
        return "Error: No record number could be found.", refresh_data

    # Retrieve the name of the animal to be deleted for confirmation message
    animal_name = selected_animal.get('name', 'Unknown')

    # Delete the animal record from the database
    deleted_count = shelter.delete({"rec_num": rec_num})

    if deleted_count > 0:
        return f"Animal record '{animal_name}' deleted successfully!", refresh_data + 1

    return "Error: Animal record could not be deleted. Please try again.", refresh_data

########################
# Geolocation Map
########################

# Updates the geolocation map for the selected
# animal in the DataTable.

@app.callback(
    Output('map-id','children'),
    [Input('datatable-id','derived_virtual_data'),
        Input('datatable-id','derived_virtual_selected_rows')]
)

def update_map(viewData, index):
    if not viewData or not index:
        return []

    dff = pd.DataFrame.from_dict(viewData)
    row = index[0]

    # Make sure selected row exists
    if row >= len(dff):
        return []

    # Get lat & long 
    latitude = dff.iloc[row].get('location_lat')
    longitude = dff.iloc[row].get('location_long')

    # If location data is missing, display message
    if pd.isna(latitude) or pd.isna(longitude):
        return [html.Div([html.H3("Location unavailable."), 
                         html.P("No location data is available for this animal.")
                     ])
                ]

    # Display map when valid coordinates available
    return [
        dl.Map(style={'width': '1000px','height': '500px'},center=[latitude,longitude], zoom=10, children=[
            dl.TileLayer( id='base-layer-id'),
            dl.Marker(position=[latitude,longitude],children=[
                dl.Tooltip(dff.iloc[row].get('name')),
                 dl.Popup([
                     html.H1( "Animal Name"),
                    html.P(dff.iloc[row].get('breed'))
                 ])
            ])
        ])
    ]



########################
# Run Application
########################

# Run app and display result.
# If port 8050 is already being used,
# an alternate port such as 8051 can be used.

app.run(debug=True)

Schema validation configured successfully.
Schema validation result: True
Validator:
{'$jsonSchema': {'bsonType': 'object', 'required': ['name', 'animal_type', 'breed', 'age_upon_outcome_in_weeks', 'sex_upon_outcome'], 'properties': {'rec_num': {'bsonType': ['int', 'long']}, 'name': {'bsonType': 'string', 'minLength': 1, 'description': 'Animal name must be a string'}, 'animal_type': {'bsonType': 'string', 'minLength': 1, 'description': 'Animal type must be a string'}, 'breed': {'bsonType': 'string', 'minLength': 1, 'description': 'Breed must be a string'}, 'age_upon_outcome_in_weeks': {'bsonType': ['int', 'long', 'double'], 'minimum': 0, 'description': 'Age must be a non-negative number'}, 'sex_upon_outcome': {'enum': ['Intact Male', 'Intact Female', 'Neutered Male', 'Spayed Female', 'Unknown'], 'description': 'Sex must contain an approved value'}}}}

Validation Level:
strict

Validation Action:
error
TEST 1 PASSED: Invalid age was rejected by MongoDB.
TEST 2 PASSED: Missing required f